# Rate Limiting
> Simple token-bucket rate limiting for FastHTML routes

In [ ]:
#| default_exp ratelimit

In [ ]:
#| export
import re,time
from math import ceil
from typing import Callable
from starlette.responses import Response
from fastcore.utils import *

In [ ]:
#| hide
from starlette.testclient import TestClient
from nbdev.showdoc import show_doc
from fastcore.test import *
from fasthtml.fastapp import *

In [ ]:
#| export
_units = dict(s=1, m=60, h=3600, d=86400)

def parse_rate(s):
    "Parse rate string like `'5/m'`, `'1 per day'`, `'100/2h'`"
    m = re.match(r'(\d+)\s*(?:/|per)\s*(\d+)?\s*([smhd])', s, re.I)
    if not m: raise ValueError(f"Invalid rate: {s}")
    n,mult,unit = m.groups()
    return int(n), _units[unit.lower()] * int(mult or 1)

`parse_rate` accepts strings in the form `"{count}/{window}"` or `"{count} per {window}"`, where:

- **count** — integer number of allowed requests (e.g. `5`, `100`)
- **window** — optional multiplier + unit: `s`econds, `m`inutes, `h`ours, `d`ays

Examples: `'5/m'`, `'100/2h'`, `'1 per day'`, `5 per 2 hours`

In [ ]:
test_eq(parse_rate('5/m'), (5, 60))
test_eq(parse_rate('100/2h'), (100, 7200))
test_eq(parse_rate('1 per day'), (1, 86400))
test_eq(parse_rate('5 per 2 hours'), (5, 7200))

In [ ]:
#| export
class TokenBucket:
    "Token-bucket rate limiter"
    def __init__(self,
                 max_reqs:str|int,     # Rate string ('5/m') or max requests per window
                 window_secs:int=None  # Window in seconds (required if `max_reqs` is int)
                ):
        if window_secs is None: max_reqs,window_secs = parse_rate(max_reqs)
        store_attr()
        self.rate = max_reqs / window_secs
        self.buckets = {}
    def __repr__(self): return f'TokenBucket({self.max_reqs}, {self.window_secs})'

    def _prune(self):
        cutoff = time.time() - self.window_secs
        self.buckets = {k:(t,ts) for k,(t,ts) in self.buckets.items() if ts > cutoff}
    def wait(self, key):
        "Return 0 if allowed, else seconds to wait"
        self._prune()
        now = time.time()
        tokens, last = self.buckets.get(key, (self.max_reqs, now))
        tokens = min(self.max_reqs, tokens + (now - last) * self.rate)
        if tokens < 1: return (1 - tokens) / self.rate
        self.buckets[key] = (tokens - 1, now)
        return 0

In [ ]:
show_doc(TokenBucket)

---

### TokenBucket

```python

def TokenBucket(
    max_reqs:str | int, # Rate string ('5/m') or max requests per window
    window_secs:int=None, # Window in seconds (required if `max_reqs` is int)
):


```

*Token-bucket rate limiter*

Implements the [token bucket algorithm](https://en.wikipedia.org/wiki/Token_bucket). Tokens are added at a steady rate and consumed by requests. When the bucket is empty, requests are rejected and told how long to wait.

You can create a bucket with either a rate string or explicit values:


In [ ]:
tb = TokenBucket(3, 10)
tb

TokenBucket(3, 10)

In [ ]:
tb = TokenBucket('3/10s')
tb

TokenBucket(3, 10)

In [ ]:
show_doc(TokenBucket.wait)

---

### TokenBucket.wait

```python

def wait(
    key
):


```

*Return 0 if allowed, else seconds to wait*

In [ ]:
test_eq(tb.wait('x'), 0)
test_eq(tb.wait('x'), 0)
test_eq(tb.wait('x'), 0)
test(tb.wait('x'), 0, operator.gt)

Each key gets its own independent token bucket. So if user A exhausts their tokens, user B is unaffected — they still have a full bucket of their own:

In [ ]:
tb2 = TokenBucket(1, 0.5)
test_eq(tb2.wait('old'), 0)
test_eq(tb2.wait('new'), 0)
test   (tb2.wait('old'), 0, operator.gt)

Stale keys — inactive longer than the window — are automatically pruned on each wait call:

In [ ]:
tb2.wait('old')
time.sleep(0.6)
tb2.wait('new')
test_eq('old' in tb2.buckets, False)
test_eq('new' in tb2.buckets, True)

Rate-limiting works by deriving a **key** from each request and giving each key its own bucket. The most common choice is to limit by client IP — `client_ip` reads it from the `X-Forwarded-For` header, falling back to `req.client.host` for direct connections.

This assumes deployment behind a single reverse proxy — specifically [Caddy](https://caddyserver.com/), which is what both [solveit](https://solve.it.com) and [plash](https://pla.sh) use. Caddy [strips any incoming `X-Forwarded-*`](https://caddyserver.com/docs/caddyfile/directives/reverse_proxy#headers) before setting its own, so clients can't spoof the IP.

**For other setups — a different reverse proxy, multi-hop (e.g. Cloudflare → Caddy), or direct exposure to the internet — write your own key function.** Trusting `X-Forwarded-For` without a stripping proxy in front lets clients trivially spoof their IP.

In [ ]:
#| export
def client_ip(req, **kwargs):
    "Get client IP from `X-Forwarded-For` header (assumes deployment behind a single Caddy reverse proxy)"
    return req.headers.get('x-forwarded-for', '').split(',')[0].strip() or (req.client and req.client.host) or ''

`limiter` ties `TokenBucket` to FastHTML's routing. It returns a `before` function — pass it as `before=limiter(...)` on a route to apply rate limiting, or reuse one `limiter(...)` across routes to share a bucket.

The `key` parameter controls what to limit by. It defaults to `client_ip` (see caveats above), but you can pass a string to match a route/query/form parameter name, or a callable `(req, **kwargs) -> str` for custom logic. 

Use `on_limit` to customize the 429 response — it receives the wait time in seconds and returns the response to send.

In [ ]:
#| export
def limiter(rate:str|tuple,             # Rate string ('5/m') or (max_reqs, window_secs) tuple
            key:str|Callable=client_ip, # Key to limit by: route param name, or callable(req, **kwargs)
            on_limit:Callable=None      # Optional callback(wait_secs) to return custom response on 429
    ):          
    "Create a `before` function that rate-limits requests"
    bucket = TokenBucket(*rate) if isinstance(rate, tuple) else TokenBucket(rate)
    def _429(w):
        if on_limit: return on_limit(w)
        return Response('Too many requests', status_code=429, headers={'Retry-After': str(ceil(w))})
    if callable(key):
        def before(req):
            if w:=bucket.wait(key(req)): return _429(w)
        return before
    async def before(req):
        v = ifnone(req.path_params.get(key), req.query_params.get(key))
        if v is None: v = (await req.form()).get(key)
        if w:=bucket.wait(str(v or '')): return _429(w)
    return before

In [ ]:
show_doc(limiter)

---

### limiter

```python

def limiter(
    rate:str | tuple, # Rate string ('5/m') or (max_reqs, window_secs) tuple
    key:Union=client_ip, # Key to limit by: route param name, or callable(req, **kwargs)
    on_limit:Callable=None, # Optional callback(wait_secs) to return custom response on 429
):


```

*Create a `before` function that rate-limits requests*

## Example usage

In [ ]:
app, rt = fast_app()
cli = TestClient(app)

### IP-based rate limiting (default)

In [ ]:
@rt(before=limiter('3/m'))  # keys on client IP by default
def index(): return 'ok'

Requests are allowed until the bucket is exhausted, then a 429 is returned with a `Retry-After` header.

In [ ]:
for i in range(3): test_eq(cli.get('/').status_code, 200)
r = cli.get('/')
test_eq(r.status_code, 429)
test(r.headers, 'Retry-After', operator.contains)

### Parameter-based rate limiting:

Different keys get independent buckets. Here we show `a@test.com` and `b@test.com` each get their own limit.

In [ ]:
@rt('/submit', methods=['POST'], before=limiter('2/m', key='email'))
def submit(email: str): return f'hello {email}'

In [ ]:
for i in range(2): test_eq(cli.post('/submit', data={'email': 'a@test.com'}).status_code, 200)
test_eq(cli.post('/submit', data={'email': 'a@test.com'}).status_code, 429)
test_eq(cli.post('/submit', data={'email': 'b@test.com'}).status_code, 200)

### Callable-based rate limiting:

For full control, pass a callable as `key`. It receives the Starlette `Request` plus any route `**kwargs`, and should return a string to bucket by. Here we rate-limit by the `x-api-key` header — each API key gets its own bucket:

In [ ]:
@rt(before=limiter('2/m', key=lambda req, **kwargs: req.headers.get('x-api-key', '')))
def custom(): return 'ok'

In [ ]:
for i in range(2): test_eq(cli.get('/custom', headers={'x-api-key': 'abc'}).status_code, 200)
test_eq(cli.get('/custom', headers={'x-api-key': 'abc'}).status_code, 429)
test_eq(cli.get('/custom', headers={'x-api-key': 'xyz'}).status_code, 200)

### Shared limits across routes:

Save the decorator and apply it to multiple routes to share a single bucket.

In [ ]:
shared = limiter('2/m')

@rt(before=shared)
def users(): return 'users'

@rt(before=shared)
def posts(): return 'posts'


In [ ]:
test_eq(cli.get('/users').status_code, 200)
test_eq(cli.get('/posts').status_code, 200)
test_eq(cli.get('/users').status_code, 429)

### Multiple limiters on one route

Pass a tuple of limiters as `before` to require all of them to pass. Below, each user gets 1 requests per minute, and the route as a whole is capped at 3 per minute per IP.

In [ ]:
@rt('/multi', before=(limiter('3/m'), limiter('1/m', key='user')))
def multi(user:str): return f'hi {user}'

In [ ]:
test_eq(cli.get('/multi?user=a').status_code, 200)
test_eq(cli.get('/multi?user=a').status_code, 429)  # user limited
test_eq(cli.get('/multi?user=b').status_code, 200)
test_eq(cli.get('/multi?user=c').status_code, 429)  # ip limited

# Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()